# Lab 06 · Multi-agent debate: propose, challenge, agree

**Prepared by Mustafa Assaf** · Lecture 3: Debate, trace, decide · ~15–20 min core work (estimate; setup and extras take longer)

## Multi-agent debate: several agents, one decision
**Start with a problem:** One architect can overlook a trade-off when choosing between API composition and CQRS.

**In this lab:** Collect independent proposals, challenge missing citations, and keep dissent. More agents add calls and do not guarantee a better answer.

**So far:** earlier labs used architecture requirements to propose service boundaries and resilience choices. Now ShopEasy support needs a screen that searches orders, payments, and deliveries by customer name .

**New:** *multi-agent debate*: several agents answer the same question, challenge each other's claims, and a fixed procedure turns their answers into one decision. Our example is an *architecture council*. Three architects with different stances, each running on a different **model family**, answer **alone and in parallel**. Then a *chair* runs the debate by fixed rules:

1. It challenges every claim that cites no source, and gives the author one chance to revise.
2. It counts votes with valid citations; a two-thirds majority decides.
3. It records the dissent, because a minority view can be right.
4. With no majority, it escalates to the human instead of guessing.

Every message is a small dictionary with a `round`, a `speaker`, and a `kind`. The list of all messages, the *transcript*, is the council's shared state: it shows who said what, based on which source.

The chair here is plain code, so its rules are easy to read and test. A chair can also be an agent, but keep its rules (who speaks, how many rounds, what counts as a majority) in code.

**At a glance**

![Lab 06 at a glance: Shared sources (R2 · R3 · case), then Minimalist (Qwen) or Safeguard (DeepSeek) or Modernizer (Gemini), then valid citations?, then Challenge + revise (one round), then Chair counts (⅔ of registered peers), then Decision (+ dissent kept) or Escalate (no majority).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/06.svg)

## Think about the example

ShopEasy support needs to search orders, payments, and deliveries by customer name. API composition joins responses; CQRS provides a searchable read model that may lag behind writes.

**Concepts:** API composition versus CQRS; Eventual consistency and read-model cost.

Use the scenario above to consider these questions before running the code:

1. Why is a customer-name search different from a lookup by order ID?
2. Which cost of CQRS must the council record?

## The idea in context

Parallel proposals preserve independent views. A coded chair limits discussion and aggregates proposals; agreement is not proof of correctness.

CQRS supports search but adds a read model and consistency lag; composition has different costs.

| Key term | Meaning |
| --- | --- |
| **Multi-agent system** | Several model-driven agents with separate roles that work on one task. |
| **Independence** | Each agent answers without seeing the others first, so its view is its own. |
| **Grounding** | Supporting a claim with a source that everyone can check. |
| **Escalation** | Handing the decision to a person when the agents cannot agree. |

## Pattern · Debate-Based Cooperation

*Multi-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Voting-Based Cooperation, Parallel Fan-Out, Shared Memory.*

| | |
| --- | --- |
| **Context** | Hard questions hold competing values and assumptions that become clearer through structured disagreement. |
| **Problem** | Independent answers never confront one another, so weak claims and hidden trade-offs survive aggregation. |
| **Solution** | Let agents with contrasting positions respond to one another over bounded rounds, using a shared transcript, then ask a judge for a structured verdict. |
| **Use when** | Trade-offs are genuinely contested; reasoning quality matters more than speed; a transcript is useful evidence. |
| **Trade-off** | Debate surfaces assumptions, but it costs many tokens, and persuasive rhetoric can outweigh truth. |
| **Flow** | Positions → Debate K rounds → Judge → Verdict |
| **Quality attributes** | Benefits: Reliability, Accuracy, Auditability. Liabilities: Latency, Cost, Accuracy, Scalability, Predictability. |
| **In this lab** | The three proposals are a *Parallel Fan-Out*, and the transcript is the debate's *Shared Memory*. The chair is a judge with fixed rules: it challenges claims without a source for one round, then aggregates votes with valid citations, as in *Voting-Based Cooperation*. Requiring sources guards against the pattern's known risk: persuasion without evidence. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Save `FREEINFERENCE_API_KEY`, `DEEPSEEK_API_KEY`, and `GEMINI_API_KEY` in Colab Secrets. Each architect uses its own provider and call budget.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.0" or not all(find_spec(name) for name in ['coursekit']):
    %pip install -q --upgrade "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/fc798d4d9fda69756f7d860756fe7c0507f7f2a2.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask

## Step 1 · Shared sources and a citation rule

All architects see the same sources: two architecture notes (R2 and R3) and two ShopEasy facts. Equal evidence keeps the debate fair. In your project, retrieve the sources with Lab 02's vector database.

`citation_valid` is a citation-validity check: a proposal may vote only if it cites at least one known source. This checks IDs, not whether a source supports a claim; students must read that claim against the cited source.

In [ ]:
sources = {  # every architect sees the same evidence, so the debate is fair
    "R2": "API composition joins small results by id; broad cross-service search is a poor fit.",
    "R3": "CQRS supports rich queries with a separate read model; the view is eventually consistent.",
    "C1": "ShopEasy support must search orders, payments and deliveries by customer name.",
    "C2": "Support accepts a slightly stale view, but the team wants few moving parts.",
}
evidence = "\n".join(f"[{sid}] {text}" for sid, text in sources.items())

def citation_valid(message):
    """Check citation IDs only; factual support still requires review."""
    return bool(message["evidence"]) and set(message["evidence"]) <= set(sources)

print(citation_valid({"evidence": ["R3", "C1"]}), citation_valid({"evidence": []}), citation_valid({"evidence": ["R99"]}))

## Step 2 · Three architects on three model families

The architects share one job and one set of sources, but each has its own stance and its own model family. The three stances are examples: in your project, design stances that fit your scenarios. Three personas on one model are not enough: one model brings one set of blind spots, so the course project requires at least three families. Each architect runs on its own provider: Qwen on FreeInference, DeepSeek, and Gemini. `PROVIDERS` in `coursekit` names the model family behind each provider, and the assert checks that the council has at least three. `asks` holds one `ask` function per architect, each with its own call budget. `say` builds every prompt the same way: the role and stance, the sources, and then the task.

Save three keys in Colab Secrets: `FREEINFERENCE_API_KEY`, `DEEPSEEK_API_KEY`, and `GEMINI_API_KEY`. This lab ignores the `PROVIDER` setting in the setup cell, because every architect has its own provider. Check each provider's pricing and quotas before class.

In [ ]:
from coursekit import PROVIDERS

council = {  # same job and sources; a different stance and model provider for each architect
    "Minimalist": {"provider": "freeinference", "stance": "Prefer the simplest design that meets the requirements."},
    "Safeguard": {"provider": "deepseek", "stance": "Prefer low-risk designs; weigh operations and failures."},
    "Modernizer": {"provider": "gemini", "stance": "Prefer designs that keep search responsive as order history grows."},
}
families = {name: PROVIDERS[a["provider"]]["family"] for name, a in council.items()}
assert len(set(families.values())) >= 3, "The council needs at least three model families."
asks = {name: make_ask(provider=a["provider"]) for name, a in council.items()}
print(families)

def say(name, task):
    """Ask one architect: its role and stance, the shared sources, then the task."""
    role = f"You are the {name} on an architecture council. {council[name]['stance']}"
    return asks[name](f"{role}\nSources:\n{evidence}\n{task}")

## Step 3 · Propose alone, in parallel

Each architect answers without seeing the others. Models tend to agree with what they have just read, so a proposal written after reading another one is not an independent opinion. `ThreadPoolExecutor` starts the three calls at the same time. (The course client limits how many requests go to one service at once. FreeInference's free tier takes one at a time, so architects that share it wait their turn; architects on different services really do run in parallel.) The chair turns each JSON reply into a message by adding the round, the speaker, and the kind.

Each proposal is made before the other proposals are shared. Step 4 challenges claims with missing or invalid citation IDs, if any occur.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor

REPLY = 'Reply with JSON only, no code fences: {"option": "...", "claim": "one sentence", "evidence": ["ids"]}'

def propose(name):
    """One architect answers alone: the prompt never contains another proposal."""
    task = f"Choose API composition or CQRS for ShopEasy support search. {REPLY}"
    return {"round": 1, "speaker": name, "kind": "proposal", **json.loads(say(name, task))}

with ThreadPoolExecutor() as pool:  # the three architects work at the same time
    proposals = {message["speaker"]: message for message in pool.map(propose, council)}
for m in proposals.values():
    print(f"{m['speaker']:<11} {m['option']:<17} {str(m['evidence']):<19} {m['claim']}")

## Step 4 · Challenge unsupported claims

The chair checks every proposal with `citation_valid`. For a claim without a source, it picks a critic, preferring an architect who chose a different option, and asks for a one-sentence challenge. The author gets one chance to revise: it can change its option, or keep it with a claim that cites sources. Allowing one challenge per claim keeps the debate bounded, so it always ends. Each message is appended to `transcript`.

The chair challenges only proposals whose citation IDs fail the check. Valid IDs do not prove that the cited text supports a claim; read the evidence yourself.

In [ ]:
transcript = list(proposals.values())  # shared state: every message, in order
for name, m in list(proposals.items()):
    if citation_valid(m):
        continue
    others = [n for n in council if n != name]
    critic = others[0]  # use the first available reviewer as a fallback
    for reviewer in others:
        if proposals[reviewer]["option"] != m["option"]:
            critic = reviewer  # prefer someone who chose a different option
            break
    challenge = say(critic, f"In one sentence, challenge this claim by the {name}: {m['claim']}")
    transcript.append({"round": 2, "speaker": critic, "kind": "challenge", "target": name, "text": challenge})
    task = f"You claimed: {m['claim']}\nChallenge: {challenge}\nRevise your proposal. {REPLY}"
    proposals[name] = {"round": 2, "speaker": name, "kind": "revision", **json.loads(say(name, task))}
    transcript.append(proposals[name])
    print(f"{critic} challenges {name}: {challenge}")
    print(f"{name} revises: {proposals[name]['option']}, {proposals[name]['claim']} {proposals[name]['evidence']}")

## Step 5 · Count votes with valid citations, keep the dissent

Only proposals with valid citations vote. With a two-thirds majority, the chair decides and keeps every dissenting proposal, with its sources, in the outcome. Without one, the chair escalates to the human. Printing the transcript replays the whole debate.

For three architects, at least two must agree. A proposal without a valid citation cannot vote, but it still counts when calculating the required majority.

In [ ]:
from collections import Counter

votes = Counter(m["option"] for m in proposals.values() if citation_valid(m))  # claims without valid citation IDs do not vote
option, count = votes.most_common(1)[0] if votes else (None, 0)
# Two-thirds of the full council must agree, including any non-voting members.
decided = count / len(council) >= 2 / 3
outcome = {"round": 3, "speaker": "Chair", "kind": "decision" if decided else "escalation",  # no guessing
           "option": option if decided else None, "votes": dict(votes),
           "dissent": [m for m in proposals.values() if decided and m["option"] != option]}
transcript.append(outcome)
for m in transcript:
    print(f"round {m['round']}  {m['speaker']:<11} {m['kind']:<10} {m.get('text') or m['option']}")
print("Votes:", outcome["votes"], "| dissent:", [(m["speaker"], m["claim"]) for m in outcome["dissent"]])

## What you should see

The chair counts valid citations and reports a decision or escalation. Proposals, challenges, vote totals, and dissent depend on the model responses.

## Try it

1. Remove citations from a proposal before the vote. Does the chair exclude it?
2. Change C1 to this scenario: low-traffic order details, lookup by id across three services. Why does R2 now favour API composition?
3. Give a false claim a real citation ID. The automated check accepts its ID; does the cited source actually support it?
4. Compare each first proposal with the final choice. Record cost and dissent even when no one changes their mind.

## Takeaway

Independent proposals, bounded critique, aggregation, and preserved dissent make an agent council inspectable. The architecture lesson is separate: API composition suits small lookups; CQRS supports rich queries but adds a read model and eventual consistency.